In [1]:
! pip install torch torchvision transformers scikit-learn seaborn


In [1]:
print("Python kernel is working")

Python kernel is working


In [2]:
! python --version

Python 3.13.9


In [3]:
! python -m pip show torch

Name: torch
Version: 2.14.1
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License-Expression: Apache-2.0 AND Apache-2.0 WITH LLVM-exception AND BSD-2-Clause AND BSD-3-Clause AND BSL-1.0 AND MIT
Location: c:\Users\ADMIN\anaconda3\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: torchvision


In [4]:
! python -X faulthandler -c "import torch; print(torch.__version__)"

OMP: Error #15: Initializing libiomp5md.dll, but found libiomp5md.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.
Fatal Python error: Aborted

Current thread 0x00002b2c (most recent call first):
  File "c:\Users\ADMIN\anaconda3\Lib\site-packages\numpy\__init__.py", line 881 in blas_fpe_check
  File "c:\Users\ADMIN\anaconda3\Lib\site-packages\numpy\__init__.py", line 89

In [5]:
! where.exe python

c:\Users\ADMIN\anaconda3\python.exe
C:\Users\ADMIN\AppData\Local\Microsoft\WindowsApps\python.exe
C:\Users\ADMIN\AppData\Local\Python\bin\python.exe


In [6]:
! python -m pip --version

pip 25.3 from c:\Users\ADMIN\anaconda3\Lib\site-packages\pip (python 3.13)



In [7]:
! conda create -n skin_ai python=3.12 pip -y

'conda' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
! conda activate skin_ai

'conda' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
! python -m pip install torch torchvision

In [10]:
! python -c "import torch; print('PyTorch:', torch.__version__); print(torch.rand(2,3))"

OMP: Error #15: Initializing libiomp5md.dll, but found libiomp5md.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.


In [11]:
! python -m pip install ipykernel

! python -m ipykernel install --user --name skin_ai --display-name "Python (skin_ai)"

Installed kernelspec skin_ai in C:\Users\ADMIN\AppData\Roaming\jupyter\kernels\skin_ai


In [3]:
! conda env list

'conda' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
! python -c "import sys; print(sys.executable)"
! python -c "import torch; print(torch.__version__)"

c:\Users\ADMIN\anaconda3\python.exe


OMP: Error #15: Initializing libiomp5md.dll, but found libiomp5md.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.


In [1]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cpu
Torchvision: 0.29.1+cpu
CUDA available: False


In [4]:
! pip install matplotlib scikit-learn seaborn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification, ViTFeatureExtractor, ViTImageProcessor
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

#Device configuration
device = torch.device("cuda" if torch.cuda.is_available()else "cpu")

#Directories
train_dir = "archive/train"
val_dir = "archive/test"

# Load Pretrained Feature Extracter
model_path = "google/vit-base-patch16-224-in21k"

feature_extractor = ViTImageProcessor.from_pretrained(model_path)

# Define image transformation(for Data Augmentation)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=feature_extractor.image_mean,std=feature_extractor.image_std)
])

# Load dataset
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

#Get Class names
class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")

#Load Pretrained ViT model
model = ViTForImageClassification.from_pretrained(
    model_path,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

model = model.to(device)


#Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

#Training the model
def train(model, train_loader, val_loader,epochs=10):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0,0,0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images).logits
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        #Validation
        model.eval()
        val_loss, correct, total = 0.0,0,0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images).logits
                loss = criterion(outputs, labels)

                val_loss += loss.item() 
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_loss = val_loss / len(val_loader)
        val_acc = val_correct / val_total

        # Append losses and accuracies
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, val_losses, train_accs, val_accs

#Train the model
train_losses, val_losses, train_accs, val_accs = train(model, train_loader, val_loader, epochs=10)





Classes: ['Acne and Rosacea Photos', 'Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions', 'Atopic Dermatitis Photos', 'Bullous Disease Photos', 'Cellulitis Impetigo and other Bacterial Infections', 'Eczema Photos', 'Exanthems and Drug Eruptions', 'Hair Loss Photos Alopecia and other Hair Diseases', 'Herpes HPV and other STDs Photos', 'Light Diseases and Disorders of Pigmentation', 'Lupus and other Connective Tissue diseases', 'Melanoma Skin Cancer Nevi and Moles', 'Nail Fungus and other Nail Disease', 'Poison Ivy Photos and other Contact Dermatitis', 'Psoriasis pictures Lichen Planus and related diseases', 'Scabies Lyme Disease and other Infestations and Bites', 'Seborrheic Keratoses and other Benign Tumors', 'Systemic Disease', 'Tinea Ringworm Candidiasis and other Fungal Infections', 'Urticaria Hives', 'Vascular Tumors', 'Vasculitis Photos', 'Warts Molluscum and other Viral Infections']


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 283.04it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "/kaggle/working/vit_model.pth")
print("Model saved successfully.")

#plot Accuracy and Loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.title("accuracy graph")

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

#Evaluate the model on the validation set
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images).logits
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

#Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

#Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


In [19]:
from transformers import ViTForImageClassification, ViTImageProcessor

model_name = "google/vit-base-patch16-224-in21k"

feature_extractor = ViTImageProcessor.from_pretrained(model_name)

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

print("Model loaded successfully!")



Loading weights: 100%|██████████| 198/198 [00:00<00:00, 277.41it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [20]:
from pathlib import Path

cache_path = Path.home() / ".cache" / "huggingface" / "hub"

print("Hugging Face cache:", cache_path)
print("Model files:")

for path in cache_path.glob("models--google--vit-base-patch16-224-in21k/snapshots/*"):
    print(path)

Hugging Face cache: C:\Users\ADMIN\.cache\huggingface\hub
Model files:
C:\Users\ADMIN\.cache\huggingface\hub\models--google--vit-base-patch16-224-in21k\snapshots\b4569560a39a0f1af58e3ddaf17facf20ab919b0


In [ ]:
import torch
import torchvision.transforms
from transformers import ViTForImageClassification, ViTFeatureExtractor
from PIL import Image

#define the image transformation
class_names = ['actinic keratoses', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']

#Load the trained model
feature_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224-in21k")

#Load the trained model
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224-in21k", num_labels=len(class_names))
model.load_state_dict(torch.load("/kaggle/working/vit_model.pth", map_location=torch.device('cpu')))
model.eval()

def process_image(image_path):
    image = Image.open(image_path).convert("RGB")
    transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std)
    ])
    image = transform(image).unsqueeze(0)  # Add batch dimension
    return image

def predict_image(image_path):
    image = process_image(image_path)
    with torch.no_grad():
        outputs = model(image).logits
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    predicted_class = class_names[predicted.item()]
    confidence_score = confidence.item() * 100 # Convert to percentage

    print(f"Predicted Class: {predicted_class}")
    print(f"Confidence: {confidence_score:.2f}%")

#Example usage:
image_path = "archive/test/actinic keratoses/ISIC_0024300.jpg"
predict_image(image_path)


